# Keyframe Interpolation Pipeline

This notebook unrolls `KeyframeInterpolationPipeline.__call__` into separate steps so every
intermediate result (prompt embeddings, conditionings, stage-1 latents) — and the transformer
weights themselves — stay cached in the kernel.

## Overview
- **Stage 1**: Generates video at half resolution using the full model
- **Stage 2**: Upsamples 2x and refines with distilled LoRA for efficiency

## Layout

Each parameter block sits directly above the first step that consumes it, so **changing a parameter
means re-running that block and everything below it** — never anything above.

| Section | Contents |
| --- | --- |
| 2 | Pipeline parameters (paths, LoRAs, quantization, offload) |
| 3 | Pipeline construction + transformer cache |
| 4 → 5 | Prompt parameters → prompt encoding |
| 6 → 7 | Sampling parameters → noiser, sigmas, guiders |
| 8 → 9 | Geometry and keyframes → stage 1 |
| 10 → 11 | Stage 2 → decode and save |
| 12 | Preview the result |
| 13 | Release the cached transformers |


## 1. Setup Imports and Configuration

In [1]:
import logging
from contextlib import ExitStack
from datetime import datetime

import torch

# Import pipeline components
from ltx_core.components.guiders import MultiModalGuiderParams, create_multimodal_guider_factory
from ltx_core.components.noisers import GaussianNoiser
from ltx_core.components.schedulers import LTX2Scheduler
from ltx_core.loader import LoraPathStrengthAndSDOps
from ltx_core.model.transformer import X0Model
from ltx_core.model.video_vae import TilingConfig, get_video_chunks_number
from ltx_core.types import VideoPixelShape

from ltx_pipelines.keyframe_interpolation import KeyframeInterpolationPipeline
from ltx_pipelines.utils.args import ImageConditioningInput
from ltx_pipelines.utils.blocks import DiffusionStage
from ltx_pipelines.utils.constants import STAGE_2_DISTILLED_SIGMAS
from ltx_pipelines.utils.denoisers import FactoryGuidedDenoiser, SimpleDenoiser
from ltx_pipelines.utils.helpers import assert_resolution, image_conditionings_by_adding_guiding_latent
from ltx_pipelines.utils.media_io import encode_video
from ltx_pipelines.utils.quantization_factory import QuantizationKind
from ltx_pipelines.utils.types import ModalitySpec, OffloadMode

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


/workspace/LTX-2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Pipeline Parameters

These define *how the models are built*. Changing anything here means re-running section 3.


In [2]:
# Paths (update these with your actual paths)
CHECKPOINT_PATH = "../../models/ltx-2.3/ltx-2.3-22b-distilled-1.1.safetensors"  # Path to main checkpoint
SPATIAL_UPSAMPLER_PATH = "../../models/ltx-2.3/ltx-2.3-spatial-upscaler-x2-1.1.safetensors"  # Path to upsampler
GEMMA_ROOT = "../../models/gemma-3-12b"  # Path to Gemma model

# LoRA configurations (optional)
LORAS: list[LoraPathStrengthAndSDOps] = []  # Applied to both stages
DISTILLED_LORA: list[LoraPathStrengthAndSDOps] = []  # Applied to stage 2 only

# Memory optimization: mirrors CLI --quantization fp8-cast --offload cpu
QUANTIZATION = QuantizationKind.FP8_CAST.to_policy(checkpoint_path=CHECKPOINT_PATH)
OFFLOAD_MODE = OffloadMode.CPU


## 3. Initialize the Pipeline

Creates the block objects (prompt encoder, conditioner, stage 1/2, upsampler, decoders) plus a small
transformer cache. Transformer weights are the ~20s-per-stage load; `stage_transformer()` keeps them
resident across runs, and `release_transformers()` frees them.

Both cells are idempotent — re-running them releases the previous transformers first.


In [3]:
_transformer_stack = ExitStack()
_transformers: dict[str, X0Model] = {}


def stage_transformer(name: str, stage: DiffusionStage) -> X0Model:
    """Load a stage's transformer once and keep it resident until `release_transformers()`."""
    if name not in _transformers:
        logger.info(f"Loading transformer '{name}' (cached until release_transformers())")
        _transformers[name] = _transformer_stack.enter_context(stage.model_context())
    return _transformers[name]


def release_transformers() -> None:
    """Tear down every cached transformer and free its pinned host / GPU buffers."""
    global _transformer_stack
    if _transformers:
        logger.info(f"Releasing transformers: {sorted(_transformers)}")
    _transformers.clear()
    _transformer_stack.close()
    _transformer_stack = ExitStack()


In [4]:
release_transformers()  # drop transformers built from a previous pipeline configuration

pipeline = KeyframeInterpolationPipeline(
    checkpoint_path=CHECKPOINT_PATH,
    distilled_lora=DISTILLED_LORA,
    spatial_upsampler_path=SPATIAL_UPSAMPLER_PATH,
    gemma_root=GEMMA_ROOT,
    loras=LORAS,
    device=device,
    quantization=QUANTIZATION,
    offload_mode=OFFLOAD_MODE,
    # compilation_config=CompilationConfig(...),  # Uncomment if needed
)
scheduler = LTX2Scheduler()

# Stage 2 only differs from stage 1 by DISTILLED_LORA, so both can share one loaded transformer.
SHARE_STAGE_TRANSFORMER = not DISTILLED_LORA

logger.info("Pipeline initialized successfully")
logger.info(f"Device: {pipeline.device}")
logger.info(f"DType: {pipeline.dtype}")
logger.info(f"Sharing one transformer across both stages: {SHARE_STAGE_TRANSFORMER}")


INFO:__main__:Pipeline initialized successfully
INFO:__main__:Device: cuda
INFO:__main__:DType: torch.bfloat16
INFO:__main__:Sharing one transformer across both stages: True


## 4. Prompt Parameters

Generation parameters are split into blocks, each sitting directly above the first step that uses it.
Change a value and re-run everything below that block — nothing above it is affected.


In [14]:
PROMPT = "Catoon style. Three childern are sitting in a bus stop. A forth child, wearing gray clothes enters from the left and sits on the bench. He says: 'you'll never believe what I found in my daddis drawer!' the child take out a gun and aims it towards the other childers. The other child shouts 'what the fuck dude!' and runs away"
NEGATIVE_PROMPT = "low quality, blurry"

# Prompt enhancement (runs inside the text encoder, before encoding)
ENHANCE_PROMPT = False
ENHANCE_PROMPT_IMAGE: str | None = None
ENHANCE_PROMPT_SEED = 42


## 5. Step 1 — Encode Prompts (cached)

Loads Gemma + the embeddings processor, encodes the positive and negative prompts, then frees them.
This is the ~30s startup cost — the resulting embeddings stay in memory, so you only pay it again
when the prompts change.


In [15]:
with torch.inference_mode():
    ctx_p, ctx_n = pipeline.prompt_encoder(
        [PROMPT, NEGATIVE_PROMPT],
        enhance_first_prompt=ENHANCE_PROMPT,
        enhance_prompt_image=ENHANCE_PROMPT_IMAGE,
        enhance_prompt_seed=ENHANCE_PROMPT_SEED,
    )

v_context_p, a_context_p = ctx_p.video_encoding, ctx_p.audio_encoding
v_context_n, a_context_n = ctx_n.video_encoding, ctx_n.audio_encoding

logger.info("Prompt embeddings cached")


INFO:ltx_pipelines.utils.blocks:Building text encoder from ../../models/gemma-3-12b
INFO:ltx_pipelines.utils.blocks:Text encoder done, building embeddings processor from ../../models/ltx-2.3/ltx-2.3-22b-distilled-1.1.safetensors
INFO:ltx_pipelines.utils.blocks:Prompt encoding complete
INFO:__main__:Prompt embeddings cached


## 6. Sampling Parameters

Seed, step count, sigma schedules and guidance scales. Re-run sections 7 → 11 after changing these.


In [16]:
SEED = 42
NUM_INFERENCE_STEPS = 50

# Sigma schedules: stage 1 falls back to the scheduler when left as None
STAGE_1_SIGMAS: torch.Tensor | None = None
STAGE_2_SIGMAS = STAGE_2_DISTILLED_SIGMAS

# Guidance parameters
VIDEO_CFG_GUIDANCE_SCALE = 7.5
VIDEO_STG_GUIDANCE_SCALE = 1.0
VIDEO_RESCALE_SCALE = 0.7
A2V_GUIDANCE_SCALE = 0.0
VIDEO_SKIP_STEP = 0
VIDEO_STG_BLOCKS: list[int] = []

AUDIO_CFG_GUIDANCE_SCALE = 7.5
AUDIO_STG_GUIDANCE_SCALE = 1.0
AUDIO_RESCALE_SCALE = 0.7
V2A_GUIDANCE_SCALE = 0.0
AUDIO_SKIP_STEP = 0
AUDIO_STG_BLOCKS: list[int] = []


## 7. Step 2 — Sampling Setup (no model loading)

Builds the noiser, sigma schedules and guider factories. Instant — re-run freely after changing
guidance scales, the seed or the sigma schedules.


In [17]:
# Re-running this cell resets the RNG stream, so stage 1 and 2 stay reproducible for a given SEED.
generator = torch.Generator(device=pipeline.device).manual_seed(SEED)
noiser = GaussianNoiser(generator=generator)

stage_1_sigmas = (
    STAGE_1_SIGMAS if STAGE_1_SIGMAS is not None else scheduler.execute(steps=NUM_INFERENCE_STEPS)
).to(dtype=torch.float32, device=pipeline.device)
stage_2_sigmas = STAGE_2_SIGMAS.to(dtype=torch.float32, device=pipeline.device)

video_guider_factory = create_multimodal_guider_factory(
    params=MultiModalGuiderParams(
        cfg_scale=VIDEO_CFG_GUIDANCE_SCALE,
        stg_scale=VIDEO_STG_GUIDANCE_SCALE,
        rescale_scale=VIDEO_RESCALE_SCALE,
        modality_scale=A2V_GUIDANCE_SCALE,
        skip_step=VIDEO_SKIP_STEP,
        stg_blocks=VIDEO_STG_BLOCKS,
    ),
    negative_context=v_context_n,
)
audio_guider_factory = create_multimodal_guider_factory(
    params=MultiModalGuiderParams(
        cfg_scale=AUDIO_CFG_GUIDANCE_SCALE,
        stg_scale=AUDIO_STG_GUIDANCE_SCALE,
        rescale_scale=AUDIO_RESCALE_SCALE,
        modality_scale=V2A_GUIDANCE_SCALE,
        skip_step=AUDIO_SKIP_STEP,
        stg_blocks=AUDIO_STG_BLOCKS,
    ),
    negative_context=a_context_n,
)

logger.info(f"Stage 1 steps: {len(stage_1_sigmas) - 1}, stage 2 steps: {len(stage_2_sigmas) - 1}")


INFO:__main__:Stage 1 steps: 50, stage 2 steps: 3


## 8. Video Geometry and Keyframes

Resolution, length and the conditioning images. Re-run sections 9 → 11 after changing these.


In [26]:
HEIGHT = 512  # Must be divisible by 8 for two-stage
WIDTH = 768  # Must be divisible by 8 for two-stage
VIDEO_LEN_SECONDS = 10
FRAME_RATE = 30.0
NUM_FRAMES = int(VIDEO_LEN_SECONDS * FRAME_RATE)
MAX_BATCH_SIZE = 1

# # Keyframes: each entry needs a frame index and blend strength
# IMAGE_PATHS = [
#     "/workspace/LTX-2/images/An anthropomorphic chimpanzee, rendered in meticulous Pix...-1.jpg",
#     "/workspace/LTX-2/images/An anthropomorphic chimpanzee, rendered in meticulous Pix...-2.jpg",
#     "/workspace/LTX-2/images/An anthropomorphic chimpanzee, rendered in meticulous Pix...-3.jpg",
# ]

IMAGE_PATHS = [
    "/workspace/LTX-2/images/busstop/Ultra-simplistic, iconic South Park cartoon style, 2D fla...-2.jpg",
    "/workspace/LTX-2/images/busstop/Ultra-simplistic, iconic South Park cartoon style, 2D fla...-3.jpg",
    "/workspace/LTX-2/images/busstop/Ultra-simplistic, iconic South Park cartoon style, 2D fla...-4.jpg",
    "/workspace/LTX-2/images/busstop/Ultra-simplistic, iconic South Park cartoon style, 2D fla...-5.jpg",
]
IMAGES = [
    ImageConditioningInput(path=IMAGE_PATHS[0], frame_idx=0, strength=1.0),
    ImageConditioningInput(path=IMAGE_PATHS[1], frame_idx=int(FRAME_RATE * 1), strength=1.0),
    ImageConditioningInput(path=IMAGE_PATHS[2], frame_idx=int(FRAME_RATE * 2), strength=1.0),
    ImageConditioningInput(path=IMAGE_PATHS[3], frame_idx=int(FRAME_RATE * 3), strength=1.0),
]

assert_resolution(height=HEIGHT, width=WIDTH, is_two_stage=True)


## 9. Step 3 — Low-Resolution Generation (Stage 1)

Encodes the keyframes into conditioning latents at half resolution, then runs the full model.
The transformer is loaded on the first run and reused afterwards, so re-running this cell only
pays for the denoising loop. Output latents stay cached for stage 2.


In [27]:
stage_1_shape = VideoPixelShape(
    batch=1, frames=NUM_FRAMES, width=WIDTH // 2, height=HEIGHT // 2, fps=FRAME_RATE
)

with torch.inference_mode():
    stage_1_conditionings = pipeline.image_conditioner(
        lambda enc: image_conditionings_by_adding_guiding_latent(
            images=IMAGES,
            height=stage_1_shape.height,
            width=stage_1_shape.width,
            video_encoder=enc,
            dtype=pipeline.dtype,
            device=pipeline.device,
        )
    )

    stage_1_video_state, stage_1_audio_state = pipeline.stage_1.run(
        stage_transformer("stage_1", pipeline.stage_1),
        denoiser=FactoryGuidedDenoiser(
            v_context=v_context_p,
            a_context=a_context_p,
            video_guider_factory=video_guider_factory,
            audio_guider_factory=audio_guider_factory,
        ),
        sigmas=stage_1_sigmas,
        noiser=noiser,
        width=stage_1_shape.width,
        height=stage_1_shape.height,
        frames=NUM_FRAMES,
        fps=FRAME_RATE,
        video=ModalitySpec(
            context=v_context_p,
            conditionings=stage_1_conditionings,
        ),
        audio=ModalitySpec(
            context=a_context_p,
        ),
        max_batch_size=MAX_BATCH_SIZE,
    )

logger.info("Stage 1 complete")


100%|██████████| 50/50 [02:57<00:00,  3.55s/it]
INFO:__main__:Stage 1 complete


## 10. Step 4 — Upsample and Refine (Stage 2)

Upsamples the stage-1 latent 2x and refines it. When `DISTILLED_LORA` is empty the stage-1
transformer is reused as-is; otherwise a second one is loaded and cached.
Re-runnable on its own as long as `stage_1_video_state` / `stage_1_audio_state` are still in memory.


In [28]:
stage_2_shape = VideoPixelShape(batch=1, frames=NUM_FRAMES, width=WIDTH, height=HEIGHT, fps=FRAME_RATE)

stage_2_transformer = (
    stage_transformer("stage_1", pipeline.stage_1)
    if SHARE_STAGE_TRANSFORMER
    else stage_transformer("stage_2", pipeline.stage_2)
)

with torch.inference_mode():
    upscaled_video_latent = pipeline.upsampler(stage_1_video_state.latent[:1])

    stage_2_conditionings = pipeline.image_conditioner(
        lambda enc: image_conditionings_by_adding_guiding_latent(
            images=IMAGES,
            height=stage_2_shape.height,
            width=stage_2_shape.width,
            video_encoder=enc,
            dtype=pipeline.dtype,
            device=pipeline.device,
        )
    )

    video_state, audio_state = pipeline.stage_2.run(
        stage_2_transformer,
        denoiser=SimpleDenoiser(v_context_p, a_context_p),
        sigmas=stage_2_sigmas,
        noiser=noiser,
        width=stage_2_shape.width,
        height=stage_2_shape.height,
        frames=NUM_FRAMES,
        fps=FRAME_RATE,
        video=ModalitySpec(
            context=v_context_p,
            conditionings=stage_2_conditionings,
            noise_scale=stage_2_sigmas[0].item(),
            initial_latent=upscaled_video_latent,
        ),
        audio=ModalitySpec(
            context=a_context_p,
            noise_scale=stage_2_sigmas[0].item(),
            initial_latent=stage_1_audio_state.latent,
        ),
    )

logger.info("Stage 2 complete")


INFO:ltx_pipelines.utils.blocks:Building video encoder + spatial upsampler from ../../models/ltx-2.3/ltx-2.3-spatial-upscaler-x2-1.1.safetensors
100%|██████████| 3/3 [00:11<00:00,  3.86s/it]
INFO:__main__:Stage 2 complete


## 11. Step 5 — Decode and Save

Decodes the stage-2 latents through the VAEs and muxes video + audio into an mp4.
`video` is a lazy iterator, so decoding actually happens while encoding.


In [29]:
tiling_config = TilingConfig.default()
video_chunks_number = get_video_chunks_number(NUM_FRAMES, tiling_config)
OUTPUT_PATH = f"output_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"

logger.info(f"Encoding video to {OUTPUT_PATH} ({video_chunks_number} chunks)")

# `video` is a lazy iterator; decoding happens inside encode_video, so both stay under inference_mode
with torch.inference_mode():
    video = pipeline.video_decoder(video_state.latent, tiling_config, generator)
    audio = pipeline.audio_decoder(audio_state.latent)

    encode_video(
        video=video,
        fps=FRAME_RATE,
        audio=audio,
        output_path=OUTPUT_PATH,
        video_chunks_number=video_chunks_number,
    )

logger.info(f"Video saved to {OUTPUT_PATH}")


INFO:__main__:Encoding video to output_20260801_151330.mp4 (6 chunks)
INFO:ltx_pipelines.utils.blocks:Building video decoder from ../../models/ltx-2.3/ltx-2.3-22b-distilled-1.1.safetensors
INFO:ltx_pipelines.utils.blocks:Building audio decoder + vocoder from ../../models/ltx-2.3/ltx-2.3-22b-distilled-1.1.safetensors
 83%|████████▎ | 5/6 [00:02<00:00,  2.37it/s]
INFO:ltx_pipelines.utils.media_io:Video saved to output_20260801_151330.mp4
INFO:__main__:Video saved to output_20260801_151330.mp4


## 12. Preview the Result

Remuxes the file written by section 11 with `+faststart` (stream copy, no re-encode) and embeds it.
Without that, the moov atom sits at the end of the file and the embedded player often loads the video
track only — which is why the clip is silent in the notebook but has sound in a desktop player.

The clip is embedded as base64, so clear this cell's output before committing the notebook.


In [ ]:
import subprocess
from pathlib import Path

from IPython.display import Video

# PyAV writes the moov atom last; the embedded player needs it up front to pick up the audio track.
PREVIEW_PATH = str(Path(OUTPUT_PATH).with_name(f"{Path(OUTPUT_PATH).stem}_preview.mp4"))
subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-i", OUTPUT_PATH, "-c", "copy", "-movflags", "+faststart", PREVIEW_PATH],
    check=True,
)

Video(PREVIEW_PATH, embed=True, width=WIDTH, html_attributes="controls loop")


## 13. Free the Cached Transformers (optional)

Run this when you are done generating, or before doing something else memory-hungry in this kernel.
The next stage run reloads the weights automatically.


In [31]:
#release_transformers()


## Reusable Workflow

Change a parameter block, then run everything below it:

| Changed | Re-run from |
| --- | --- |
| paths, LoRAs, quantization, offload (2) | 3 |
| prompts / enhancement (4) | 5 |
| seed, steps, sigmas, guidance (6) | 7 |
| resolution, length, keyframes (8) | 9 |
| output path only | 11 |

Cached kernel state after a full run:

- `pipeline`, `scheduler`, and the loaded transformers behind `stage_transformer()` (section 3)
- `ctx_p` / `ctx_n`, `v_context_p`, `a_context_p`, `v_context_n`, `a_context_n` — prompt embeddings (section 5)
- `video_guider_factory`, `audio_guider_factory`, `stage_1_sigmas`, `stage_2_sigmas`, `noiser` (section 7)
- `stage_1_video_state`, `stage_1_audio_state`, `stage_1_conditionings` (section 9)
- `video_state`, `audio_state`, `upscaled_video_latent` (section 10)
- `OUTPUT_PATH` — the mp4 played back by section 12

The transformer is loaded once — and shared by both stages while `DISTILLED_LORA` is empty — so repeat
runs skip the ~20s-per-stage weight load. Its blocks stay in pinned host RAM (streaming to GPU per block),
while the text encoder, VAEs and upsampler are still built and freed per call. Section 13 releases it;
re-running section 3 does so automatically.
